# Librerias

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
from gensim.models import KeyedVectors, Word2Vec
import gdown
import ast
from sentence_transformers import SentenceTransformer, util
from prettytable import PrettyTable
import unicodedata
import re
from collections import Counter

# Apertura del csv

In [9]:
def abrir_archivo_csv(ruta_csv):
    """Abre un archivo CSV y devuelve un DataFrame de pandas."""
    try:
        df = pd.read_csv(ruta_csv)
        return df
    except FileNotFoundError:
        print(f"Error: No se encontró el archivo {ruta_csv}")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: El archivo {ruta_csv} está vacío")
        return None
    except pd.errors.ParserError:
        print(f"Error: No se pudo parsear el archivo {ruta_csv}")
        return None

# Limpieza del dataset

In [10]:
def encontrar_raiz_proyecto():
    """
    Devuelve la carpeta que contiene a "Practica1" y "Practica 2", sin importar desde
    donde se ejecute el notebook.
    Orden de busqueda:
      1. Variable de entorno NLP_PROYECTO (si esta definida).
      2. El directorio actual, sus padres y las subcarpetas directas de cada uno.
    """
    def es_raiz(carpeta):
        return (carpeta / "Practica1" / "data" / "libros.csv").is_file()

    entorno = os.environ.get("NLP_PROYECTO")
    if entorno and es_raiz(Path(entorno)):
        return Path(entorno).resolve()

    actual = Path.cwd().resolve()
    for base in [actual, *actual.parents]:
        if es_raiz(base):
            return base
        try:
            for sub in base.iterdir():
                if sub.is_dir() and es_raiz(sub):
                    return sub
        except OSError:
            continue

    raise FileNotFoundError(
        "No se encontro 'Practica1/data/libros.csv'. Ejecuta el notebook dentro del proyecto "
        "o define la variable de entorno NLP_PROYECTO con la carpeta que contiene Practica1 y Practica 2."
    )

In [11]:
RUTA_PROYECTO = encontrar_raiz_proyecto()
RUTA_CSV_ORIGINAL = RUTA_PROYECTO / "Practica1" / "data" / "libros.csv"
RUTA_CSV_LIMPIO = RUTA_PROYECTO / "Practica 2" / "data" / "libros_limpios.csv"
RUTA_MODELO = RUTA_PROYECTO / "Practica 2" / "modelos"

RUTA_CSV_LIMPIO.parent.mkdir(parents=True, exist_ok=True)
RUTA_MODELO.mkdir(parents=True, exist_ok=True)

In [ ]:
STOPWORDS_ES = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "como",
    "con", "contra", "cual", "cuando", "de", "del", "desde", "donde",
    "dos", "el", "ella", "ellas", "ellos", "en", "entre", "era", "eran",
    "es", "esa", "esas", "ese", "eso", "esos", "esta", "estas", "este",
    "esto", "estos", "fue", "ha", "han", "hasta", "la", "las", "le", "les",
    "lo", "los", "más", "me", "mi", "mis", "muy", "ni", "no", "nos",
    "o", "otra", "otras", "otro", "otros", "para", "pero", "por", "que",
    "se", "sin", "sobre", "su", "sus", "también", "te", "tiene", "tienen",
    "tu", "tus", "un", "una", "unas", "uno", "unos", "y", "ya",
}

COLUMNAS_DE_TEXTO = [
    "titulo",
    "serie",
    "sinopsis",
]


def minusculas(texto):
    """
    Convierte un texto a minúsculas.
    Args: texto (str): Texto a convertir.
    Returns: str: Texto en minúsculas.
    """
    return texto.lower()


def acentos_dieresis_virgulilla(texto):
    """
    Elimina los acentos de un texto, dieresis en la letra 'u' y virgulilla en la letra 'ñ'.
    Args: texto (str): Texto a procesar.
    Returns: str: Texto sin acentos.
    """
    # Se normaliza caracter por caracter: si se aplicara NFD al texto completo, la 'ñ' se
    # descompondria en 'n' + virgulilla y se perderia al quitar las marcas.
    resultado = []
    for c in texto:
        if c in 'ñÑ':
            resultado.append(c)
        else:
            descompuesto = unicodedata.normalize('NFD', c)
            resultado.extend(d for d in descompuesto if unicodedata.category(d) != 'Mn')
    return ''.join(resultado)


def puntuacion(texto):
    """
    Elimina la puntuación de un texto (y otros signos)
    Args: texto (str): Texto a procesar.
    Returns: str: Texto sin puntuación.
    """
    return ''.join(c for c in texto if c.isalnum() or c.isspace())


def stopwords(texto, stopwords_set):
    """
    Elimina las stopwords de un texto.
    Args: texto (str): Texto a procesar.
        stopwords_set (set): Conjunto de stopwords a eliminar.
    Returns: str: Texto sin stopwords.
    """
    return ' '.join(palabra for palabra in texto.split() if palabra not in stopwords_set)


# Las stopwords se normalizan con el mismo proceso que el texto (minusculas y sin acentos);
# si no, palabras como "más" o "también" nunca coincidirian con el texto ya sin acentos.
STOPWORDS_ES = {acentos_dieresis_virgulilla(minusculas(palabra)) for palabra in STOPWORDS_ES}


def limpiar_texto(texto):
    """Aplica todas las funciones de procesamiento de texto en orden."""
    texto = minusculas(texto)
    texto = acentos_dieresis_virgulilla(texto)
    texto = puntuacion(texto)
    texto = stopwords(texto, STOPWORDS_ES)
    return texto


def limpiar_libros():
    """Copia el CSV original, limpia sus columnas textuales y crea otro CSV."""
    if not RUTA_CSV_ORIGINAL.exists():
        raise FileNotFoundError(
            f"No se encontró el CSV original: {RUTA_CSV_ORIGINAL}"
        )

    libros = pd.read_csv(RUTA_CSV_ORIGINAL, keep_default_na=False)
    columnas_faltantes = [
        columna for columna in COLUMNAS_DE_TEXTO if columna not in libros.columns
    ]
    if columnas_faltantes:
        raise ValueError(f"Faltan columnas de texto: {columnas_faltantes}")

    for columna in COLUMNAS_DE_TEXTO:
        libros[columna] = libros[columna].map(limpiar_texto)

    libros["autores"] = libros["autores"].apply(ast.literal_eval)
    libros["generos"] = libros["generos"].apply(ast.literal_eval)
    libros["autores"] = libros["autores"].apply(lambda autores: list(map(limpiar_texto, autores)))
    libros["generos"] = libros["generos"].apply(lambda generos: list(map(limpiar_texto, generos)))

    libros.to_csv(RUTA_CSV_LIMPIO, index=False, encoding="utf-8")
    print(f"CSV limpio creado: {RUTA_CSV_LIMPIO}")
    print(f"Registros procesados: {len(libros)}")

In [14]:
limpiar_libros()

NameError: name 'minusculas' is not defined

# Elección de modelo

In [ ]:
vector_sizes = [50, 100, 200]
windows = [2, 4, 6]
min_counts = [1, 2, 5]
sg = [0,1]  # 0: CBOW, 1: Skip-gram

In [ ]:
def word2vec(
    sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0
):
    """
    Crea y entrena un modelo Word2Vec.

    Arguments:
        sentences: Lista de listas de palabras.
        vector_size: Dimensionalidad de los vectores.
        window: Tamaño de la ventana de contexto.
        min_count: Mínimo número de ocurrencias de una palabra.
        workers: Número de hilos utilizados.
        sg: Tipo de modelo:
            0 = CBOW
            1 = Skip-gram.

    Returns:
        Word2Vec: Modelo Word2Vec entrenado.
    """

    modelo_word2vec = Word2Vec(
        sentences=sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=sg,
        seed=42
    )

    return modelo_word2vec

## Tokenizar

In [ ]:
def tokenizar(dataset: pd.DataFrame, columna: str):
    """
    Tokeniza el texto en la columna especificada del DataFrame.

    Arguments:
        dataset: pd.DataFrame - DataFrame que contiene la columna.
        columna: str - Nombre de la columna que se desea tokenizar.

    Returns:
        pd.Series - Serie de pandas con los textos tokenizados.
    """

    return dataset[columna].apply(
        lambda x: str(x).split()
    )

## Evaluar

In [ ]:
def evaluar_pares(modelo, pares):
    """
    Calcula la similitud promedio entre pares de palabras.

    Solo se utilizan los pares cuyas dos palabras
    están presentes en el vocabulario del modelo.

    Arguments:
        modelo: Modelo Word2Vec entrenado.
        pares: Lista de tuplas (palabra1, palabra2).

    Returns:
        float: Similitud promedio.
    """

    resultados = []

    for palabra1, palabra2 in pares:

        if palabra1 in modelo.wv and palabra2 in modelo.wv:

            similitud = modelo.wv.similarity(
                palabra1,
                palabra2
            )

            resultados.append(similitud)

    if not resultados:
        return None

    return np.mean(resultados)

## Evaluar Modelos

In [ ]:
def evaluar_vecinos(modelo, palabras, topn=5):
    """
    Obtiene los vecinos más similares de una lista de palabras.

    Arguments:
        modelo: Modelo Word2Vec o KeyedVectors.
        palabras: Lista de palabras.
        topn: Cantidad de vecinos a devolver.

    Returns:
        dict: Diccionario donde cada palabra tiene asociados sus vecinos
              más similares y su similitud.
    """

    resultados = {}

    for palabra in palabras:

        if palabra not in modelo.key_to_index:
            resultados[palabra] = None
            continue

        vecinos = modelo.most_similar(palabra, topn=topn)

        resultados[palabra] = vecinos

    return resultados

In [ ]:
def evaluar_modelos(modelos_word2vec, pares):
    """
    Evalúa todos los modelos Word2Vec entrenados utilizando los pares de palabras.

    Arguments:
        modelos_word2vec: Diccionario con los modelos Word2Vec entrenados.
        pares: Lista de tuplas (palabra1, palabra2).

    Returns:
        dict: Diccionario con la similitud promedio para cada modelo.
    """

    promedios = {}

    for nombre_modelo, modelo in modelos_word2vec.items():
        print(f"Evaluando modelo {nombre_modelo}")
        resultados = evaluar_pares(modelo, pares)
        if resultados is not None:
            promedios[nombre_modelo] = resultados

        else:
            print("No se encontraron palabras en el vocabulario del modelo.")

    return promedios

## Entrenamiento de modelos

In [ ]:
def entrenar_modelo_word2vec(
    sentences: pd.DataFrame,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=0
):
    """
    Entrena un modelo Word2Vec utilizando la columna especificada.

    Arguments:
        sentences: pd.DataFrame - DataFrame que contiene las oraciones ya tokenizadas.
        vector_size: int - Dimensionalidad de los vectores.
        window: int - Tamaño de la ventana de contexto.
        min_count: int - Mínimo número de ocurrencias.
        workers: int - Número de hilos.
        sg: int - Tipo de modelo:
            0 = CBOW
            1 = Skip-gram.

    Returns:
        Word2Vec - Modelo Word2Vec entrenado.
    """

    modelo_word2vec = word2vec(
        sentences,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        workers=workers,
        sg=sg
    )

    return modelo_word2vec

In [ ]:
def entrenar_modelos():

    libros = pd.read_csv(RUTA_CSV_LIMPIO)

    modelos_word2vec = {}

    # Tokenizamos una sola vez.
    # No es necesario hacerlo en cada iteración.
    sentences = tokenizar(
        libros,
        "sinopsis"
    )

    # Probamos diferentes hiperparámetros
    for vector_size in vector_sizes:

        for window in windows:

            for min_count in min_counts:

                for sg_value in sg:

                    print(
                        f"Entrenando modelo con "
                        f"vector_size={vector_size}, "
                        f"window={window}, "
                        f"min_count={min_count}, "
                        f"sg={sg_value}"
                    )

                    modelo_word2vec = entrenar_modelo_word2vec(
                        sentences,
                        vector_size=vector_size,
                        window=window,
                        min_count=min_count,
                        sg=sg_value
                    )

                    nombre_modelo = (
                        f"modelo_word2vec_"
                        f"{vector_size}_"
                        f"{window}_"
                        f"{min_count}_"
                        f"{sg_value}"
                    )

                    modelos_word2vec[
                        nombre_modelo
                    ] = modelo_word2vec

    return modelos_word2vec

## Seleción del modelo

In [ ]:
def seleccionar_mejor_modelo(modelos_word2vec, pares):
    """
    Selecciona el mejor modelo Word2Vec basado en la similitud promedio
    de los pares de palabras.

    Arguments:
        modelos_word2vec: Diccionario con los modelos Word2Vec entrenados.
        pares: Lista de tuplas (palabra1, palabra2).

    Returns:
        tuple: (nombre del mejor modelo, similitud promedio)
    """

    promedios = evaluar_modelos(modelos_word2vec, pares)

    if not promedios:
        print("No se encontraron palabras en el vocabulario de los modelos.")
        return None, None

    mejor_modelo = max(promedios, key=promedios.get)
    mejor_similitud = promedios[mejor_modelo]

    return mejor_modelo, mejor_similitud

## Guardar modelo y metricas

In [ ]:
def guardar_modelo(modelo, nombre_modelo):
    """
    Guarda el modelo Word2Vec en un archivo.

    Arguments:
        modelo: Modelo Word2Vec entrenado.
        nombre_modelo: Nombre del archivo donde se guardará el modelo.
    """
    ruta_modelo = str(RUTA_MODELO / f"{nombre_modelo}.model")

    modelo.save(ruta_modelo)
    print(f"Modelo guardado en: {ruta_modelo}")

def guardar_combinacion_parametros(nombre_modelo):
    """
    Guarda la combinacion de parámetros del mejor modelo en un archivo de texto.

    Arguments:
        nombre_modelo: Nombre del archivo donde se guardará la combinacion de parámetros.
    """
    ruta_parametros = RUTA_MODELO / "mejor_modelo_parametros.txt"
    with open(ruta_parametros, "w", encoding="utf-8") as f:
        f.write(
            f"Mejor modelo: {nombre_modelo}\n"
            f"Parámetros: {nombre_modelo.split('_')[2:]}\n"
        )
    print(f"Combinación de parámetros guardada en: {ruta_parametros}")

## Ejecución

In [ ]:
def elegir_mejor_modelo():
    modelos = entrenar_modelos()

    pares = [
        ("abuelo", "nieto"),
        ("padre", "hijo"),
        ("rey", "reina"),
        ("guerra", "batalla"),
        ("amor", "corazon"),
        ("viaje", "aventura"),
    ]

    mejor_modelo, mejor_similitud = seleccionar_mejor_modelo(modelos, pares)

    if mejor_modelo:
        print(f"El mejor modelo es: {mejor_modelo}, con una similitud promedio de: {mejor_similitud}")
        guardar_modelo(modelos[mejor_modelo], mejor_modelo)
        guardar_combinacion_parametros(mejor_modelo)
    else:
        print("No se pudo determinar el mejor modelo.")

    return modelos

In [ ]:
modelos = elegir_mejor_modelo()

## Selección de modelo con pares negativos (versión alternativa)

La selección original promedia la similitud de 6 pares relacionados y descarta en silencio los pares con palabras fuera del vocabulario, por lo que cada modelo se promedia sobre pares distintos. Esta versión alternativa, que se muestra **junto** a la original para comparar, hace tres cosas:

1. Usa solo los pares cuyas palabras están en el vocabulario de **todos** los modelos, así los puntajes son comparables.
2. Agrega pares **negativos** (palabras no relacionadas) y puntúa con `similitud de positivos - similitud de negativos`. Un modelo donde todo se parece a todo obtiene un contraste cercano a 0.
3. Informa cuántos pares se usaron.

In [ ]:
PARES_POSITIVOS = [
    ("abuelo", "nieto"),
    ("padre", "hijo"),
    ("rey", "reina"),
    ("guerra", "batalla"),
    ("amor", "corazon"),
    ("viaje", "aventura"),
]

PARES_NEGATIVOS = [
    ("abuelo", "batalla"),
    ("rey", "viaje"),
    ("padre", "aventura"),
    ("nieto", "guerra"),
    ("reina", "hijo"),
    ("corazon", "batalla"),
]


def vocabulario_comun(modelos_word2vec):
    """Devuelve el conjunto de palabras presentes en el vocabulario de todos los modelos."""
    vocabularios = [set(modelo.wv.key_to_index) for modelo in modelos_word2vec.values()]
    return set.intersection(*vocabularios)


def filtrar_pares(pares, vocabulario):
    """Se queda solo con los pares cuyas dos palabras estan en el vocabulario."""
    return [(a, b) for a, b in pares if a in vocabulario and b in vocabulario]


def seleccionar_mejor_modelo_contraste(modelos_word2vec, pares_positivos, pares_negativos):
    """
    Evalua los modelos con pares positivos y negativos, usando solo pares presentes
    en el vocabulario de todos los modelos.

    Returns:
        tuple: (lista de filas ordenada por contraste, pares positivos usados, pares negativos usados)
    """
    vocabulario = vocabulario_comun(modelos_word2vec)
    positivos = filtrar_pares(pares_positivos, vocabulario)
    negativos = filtrar_pares(pares_negativos, vocabulario)

    if not positivos or not negativos:
        raise ValueError("No quedan pares positivos y negativos en el vocabulario comun.")

    filas = []
    for nombre_modelo, modelo in modelos_word2vec.items():
        sim_pos = evaluar_pares(modelo, positivos)
        sim_neg = evaluar_pares(modelo, negativos)
        filas.append((nombre_modelo, sim_pos, sim_neg, sim_pos - sim_neg))

    filas.sort(key=lambda fila: fila[3], reverse=True)
    return filas, positivos, negativos

In [ ]:
filas, positivos_usados, negativos_usados = seleccionar_mejor_modelo_contraste(
    modelos, PARES_POSITIVOS, PARES_NEGATIVOS
)

print(f"Pares positivos usados: {len(positivos_usados)} de {len(PARES_POSITIVOS)}")
print(f"Pares negativos usados: {len(negativos_usados)} de {len(PARES_NEGATIVOS)}")

tabla = PrettyTable()
tabla.field_names = ["Modelo", "Sim. positivos", "Sim. negativos", "Contraste"]
for nombre_modelo, sim_pos, sim_neg, contraste in filas[:10]:
    tabla.add_row([nombre_modelo, f"{sim_pos:.4f}", f"{sim_neg:.4f}", f"{contraste:.4f}"])
print("Top 10 modelos segun el contraste:")
print(tabla)

mejor_original, similitud_original = seleccionar_mejor_modelo(modelos, PARES_POSITIVOS)
mejor_contraste = filas[0][0]
print(f"Mejor modelo (criterio original, solo positivos): {mejor_original} ({similitud_original:.4f})")
print(f"Mejor modelo (criterio con contraste):            {mejor_contraste} ({filas[0][3]:.4f})")
print("Coinciden." if mejor_original == mejor_contraste else "Los criterios eligen modelos distintos.")

# Word2Vec

## Descargar modelo SBW

In [ ]:
# ruta_modelo = 'NLP/Practica 2/modelos/SBW-vectors-300-min5.bin.gz'
url = "https://drive.google.com/uc?id=1V8hNcnGEyrz0c_dA-v0_5sg31bNgy75r"

def descarga_modelo(ruta_modelo):
    """
    Descarga el modelo SBW (Spanish Billion Words) desde Google Drive y lo guarda en la ruta especificada.
    Crea los directorios necesarios si no existen.
    """
    os.makedirs(os.path.dirname(ruta_modelo), exist_ok=True)
    gdown.download(url, ruta_modelo, quiet=False)

In [ ]:
def SBW_modelo(ruta_modelo):
    """
    Carga el modelo SBW (Spanish Billion Words) desde la ruta especificada.
    Si el modelo no existe, lo descarga primero.
    Returns:
        KeyedVectors: El modelo SBW cargado.
    """
    if not os.path.exists(ruta_modelo):
        descarga_modelo(ruta_modelo)

    model = KeyedVectors.load_word2vec_format(ruta_modelo, binary=True)
    return model

## Parametros

In [ ]:
def parametros_word2vec(ruta_parametros):
    with open(ruta_parametros, "r", encoding="utf-8") as archivo:
        lineas = archivo.readlines()
        for linea in lineas:
            print(linea.strip())

    modelo, parametros = None, None

    for linea in lineas:
        if "Mejor modelo" in linea:
            modelo = linea.split(":")[1].strip()

        if "Parámetros" in linea:
            parametros = linea.split(":")[1].strip()

    parametros = ast.literal_eval(parametros)
    return modelo, parametros

## Palabras del dominio

In [ ]:
def palabras_dominio(libros):
    palabras = []

    for texto in libros["sinopsis"]:
        palabras.extend(str(texto).split())

    frecuencias = Counter(palabras)

    return [palabra for palabra, frecuencia in frecuencias.most_common(4)]

## Excepcion

In [ ]:
class ModeloNoEncontradoError(FileNotFoundError):
    """Se lanza cuando el modelo Word2Vec indicado no existe en disco."""
    pass

## Bucle principal para esta sección

In [ ]:
def bucle_principal(ruta_csv, ruta_parametros, ruta_modelos):
    libros = abrir_archivo_csv(ruta_csv)

    modelo, _ = parametros_word2vec(ruta_parametros)
    ruta_modelo_guardado = os.path.join(ruta_modelos, modelo + ".model")
    if not os.path.exists(ruta_modelo_guardado):
        raise ModeloNoEncontradoError(
            f"No se encontro el modelo Word2Vec guardado en: {ruta_modelo_guardado}. "
            "Ejecuta primero la eleccion de modelo."
        )
    modelo_word2vec = Word2Vec.load(ruta_modelo_guardado)

    ruta_sbw = os.path.join(ruta_modelos, "SBW-vectors-300-min5.bin.gz")

    modelo_sbw = SBW_modelo(ruta_sbw)

    palabras_frecuentes = palabras_dominio(libros)

    resultados_word2vec = evaluar_vecinos(modelo_word2vec.wv, palabras_frecuentes, topn=5)
    resultados_sbw = evaluar_vecinos(modelo_sbw,palabras_frecuentes, topn=5)

    print("Resultados Word2Vec:")
    for palabra, vecinos in resultados_word2vec.items():
        print(f"  {palabra}: {vecinos}")

    print("Resultados SBW:")
    for palabra, vecinos in resultados_sbw.items():
        print(f"  {palabra}: {vecinos}")

In [ ]:
bucle_principal(RUTA_CSV_LIMPIO, RUTA_MODELO / "mejor_modelo_parametros.txt", RUTA_MODELO)

El modelo Word2Vec entrenado sobre el corpus de libros devuelve similitudes coseno casi iguales a 1 (≈ 0,999) para todos sus vecinos, mientras que SBW devuelve valores entre 0,58 y 0,79. Estos valores no son directamente comparables, porque provienen de espacios vectoriales entrenados sobre corpus distintos. Además, que todos los vecinos tengan prácticamente la misma similitud indica que el corpus propio (200 sinopsis) es demasiado pequeño: los vectores quedan casi idénticos entre sí y el ranking de vecinos es poco confiable.

Esto se ve en las palabras recuperadas. Con Word2Vec, "novela" devuelve "nunca", "universo", "lugar", "vidas" y "son"; "historia" devuelve "san", "primera", "estan", "durante" y "cuenta"; "aventuras" devuelve "tanto", "pais", "autor", "nunca" e "imposible"; y "mundo" devuelve "destino", "hacer", "sera", "san" y "alli". Casi ninguna tiene una relación semántica directa con la palabra consultada (solo "autor" o "destino" tienen alguna relación con el dominio literario). Como el entrenamiento usa varios hilos, estos vecinos pueden variar levemente entre ejecuciones, pero la conclusión se mantiene.

SBW, en cambio, recupera relaciones semánticas claras del español: para "novela" aparecen "novelas", "cuento", "narrativa" y "relato"; para "aventuras", "desventuras", "peripecias" y "andanzas"; para "historia", "contada" y "narración"; y para "mundo", "orbe", "mundial", "país" y "continente".

Por lo tanto, con el tamaño actual del corpus, SBW representa mucho mejor las relaciones semánticas de las palabras consultadas. La posible ventaja del modelo propio, que es conocer el vocabulario específico del dominio, no se aprecia en estos resultados. La comparación muestra la diferencia entre un modelo entrenado sobre un corpus pequeño y uno preentrenado sobre un corpus mucho más amplio.

Respecto de la selección del modelo, el criterio original (solo pares relacionados) elige `modelo_word2vec_200_6_5_1` con una similitud promedio de ≈ 0,9989, un valor que no discrimina porque casi todo se parece a todo. El criterio alternativo, que resta la similitud de pares no relacionados, elige un modelo distinto (`modelo_word2vec_100_2_1_1` en nuestra corrida) con un contraste de ≈ 0,10. Esto sugiere que la selección original premia modelos poco discriminativos.

# Modelo de oraciones

In [ ]:
def separar_oraciones(texto, min_palabras=5):
    """
    Separa un texto en oraciones (cortando en . ! ? y …) y descarta las muy cortas.
    """
    oraciones = re.split(r"(?<=[.!?…])\s+", str(texto))
    return [o.strip() for o in oraciones if len(o.split()) >= min_palabras]


def modelo_oraciones(ruta_csv_original, topn=10):
    """
    Busca los pares de oraciones mas similares entre todas las sinopsis del CSV original.
    Se usa el CSV original (con puntuacion y mayusculas) porque la limpieza elimina los
    puntos y no permitiria separar oraciones. Usa util.paraphrase_mining, que compara por
    bloques y solo conserva los mejores vecinos, sin armar la matriz completa N x N.
    """
    # --- Como se reduce el consumo de recursos ---
    # Comparar todas las oraciones contra todas (N oraciones) implica N x N similitudes.
    # Con las 1135 oraciones actuales serian ~1,3 millones de valores, pero con un dataset
    # grande (por ejemplo 100.000 oraciones) serian 10.000 millones de valores (decenas de GB
    # de RAM solo para la matriz) y ademas habria que guardar y ordenar la lista de todos los pares.
    #
    # La version anterior hacia justamente eso:
    #   1. util.cos_sim(embeddings, embeddings)  -> matriz completa N x N en memoria.
    #   2. Una lista con los N*(N-1)/2 pares (i, j) -> otra estructura que tambien crece con N^2.
    #   3. sorted() sobre esa lista -> ordenar millones de elementos solo para quedarnos con 10.
    #
    # util.paraphrase_mining evita esto de la siguiente manera:
    #   - Calcula los embeddings una sola vez (N x 512 valores: crece linealmente con N).
    #   - Compara por BLOQUES: toma query_chunk_size oraciones y las compara contra
    #     corpus_chunk_size oraciones a la vez, de modo que solo existe en memoria una
    #     sub-matriz pequena (query_chunk_size x corpus_chunk_size) en cada paso.
    #   - De cada bloque conserva unicamente los top_k vecinos mas cercanos de cada oracion y
    #     descarta el resto, asi que lo que se guarda es del orden de N * top_k pares y no N^2.
    #   - Recien al final ordena esa lista reducida, que es mucho mas chica.
    # Resultado: la memoria crece con N (y no con N^2) y el tiempo de ordenamiento baja mucho.
    # Los pares del top 10 global se conservan igual: si (i, j) esta entre los mejores del
    # dataset, j tambien esta entre los mejores vecinos de i, por eso top_k=5 es suficiente.
    #
    # Parametros que controlan el consumo:
    #   - query_chunk_size / corpus_chunk_size: tamano de cada bloque. Bloques mas chicos
    #     usan menos RAM (pero tardan un poco mas).
    #   - top_k: cuantos vecinos se conservan por oracion. Menor top_k, menos memoria.
    #   - batch_size: cuantas oraciones se pasan juntas por la red al calcular embeddings.
    #     Menor batch_size, menos memoria durante la inferencia.
    #
    # Formas de bajarlo aun mas:
    #   - Pedir los embeddings como arreglo de NumPy (convert_to_numpy=True) en lugar de tensor
    #     de PyTorch si se trabaja solo en CPU, evitando copias.
    #   - Usar un modelo de embeddings mas chico (menos parametros y menos dimensiones por
    #     vector, por ejemplo 384 en lugar de 512), a costa de algo de calidad.
    #   - Comparar solo entre oraciones de distintos libros, o filtrar oraciones, para reducir N.
    libros = pd.read_csv(ruta_csv_original, keep_default_na=False)

    # dict.fromkeys elimina oraciones repetidas conservando el orden
    oraciones = list(dict.fromkeys(
        oracion for sinopsis in libros["sinopsis"] for oracion in separar_oraciones(sinopsis)
    ))
    print(f"Oraciones a comparar: {len(oraciones)}")

    modelo = SentenceTransformer("distiluse-base-multilingual-cased-v1")
    pares = util.paraphrase_mining(
        modelo,
        oraciones,
        top_k=5,
        query_chunk_size=500,
        corpus_chunk_size=2000,
        batch_size=32,
    )

    tabla = PrettyTable()
    tabla.field_names = ["Oración 1", "Oración 2", "Puntuación de Similitud"]
    tabla.max_width = 60

    for score, i, j in pares[:topn]:
        tabla.add_row([oraciones[i], oraciones[j], f"{score:.4f}"])

    print(tabla)

In [ ]:
modelo_oraciones(RUTA_CSV_ORIGINAL)